In [1]:
import os
import shutil
import random
import pathlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from pathlib import Path

PLANTVILLAGE = Path("/content/PlantVillage-Dataset")

if not (PLANTVILLAGE / "raw" / "color").exists():
    !git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git

COLOR_DIR = PLANTVILLAGE / "raw" / "color"

# Download CIFAR-10 through Keras.
(x_cifar, y_cifar), (x_cifar_test, y_cifar_test) = (
    tf.keras.datasets.cifar10.load_data()
)

# Download TensorFlow's flower photos.
flower_archive = tf.keras.utils.get_file(
    "flower_photos.tgz",
    origin=(
        "https://storage.googleapis.com/"
        "download.tensorflow.org/example_images/flower_photos.tgz"
    ),
    extract=True
)

FLOWER_DIR = Path(flower_archive).parent / "flower_photos"

print("PlantVillage:", COLOR_DIR)
print("CIFAR-10 training images:", len(x_cifar))
print("Flower directory:", FLOWER_DIR)

Cloning into 'PlantVillage-Dataset'...
remote: Enumerating objects: 163219, done.
remote: Counting objects: 100% (163219/163219), done.
remote: Compressing objects: 100% (163125/163125), done.
remote: Total 163219 (delta 93), reused 163215 (delta 93), pack-reused 0 (from 0)
Receiving objects: 100% (163219/163219), 2.00 GiB | 36.28 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Updating files: 100% (182404/182404), done.
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 863s 5us/step
228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
PlantVillage: /content/PlantVillage-Dataset/raw/color
CIFAR-10 training images: 50000
Flower directory: /root/.keras/datasets/flower_photos


In [3]:
DATA_DIR = Path("/content/tomato_validator_data")

for split in ["train", "val", "test"]:
    for label in ["leaf", "non_leaf"]:
        (DATA_DIR / split / label).mkdir(
            parents=True, exist_ok=True
        )

tomato_classes = [
    "Tomato___healthy",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
]

positive_files = []

for class_name in tomato_classes:
    class_dir = COLOR_DIR / class_name
    if not class_dir.exists():
        raise FileNotFoundError(class_dir)

    positive_files.extend([
        p for p in class_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])

random.shuffle(positive_files)
positive_files = positive_files[:2500]

# CIFAR-10: sample objects, animals, vehicles, etc.
negative_arrays = list(x_cifar) + list(x_cifar_test)
random.shuffle(negative_arrays)
negative_arrays = negative_arrays[:2000]

# Add flower photos as additional plant-related negatives.
flower_files = [
    p for p in FLOWER_DIR.rglob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]
random.shuffle(flower_files)
flower_files = flower_files[:500]

negative_count = len(negative_arrays) + len(flower_files)

print("Positive tomato leaf images:", len(positive_files))
print("Negative object images:", len(negative_arrays))
print("Negative flower images:", len(flower_files))
print("Total negatives:", negative_count)

Positive tomato leaf images: 2500
Negative object images: 2000
Negative flower images: 0
Total negatives: 2000


In [5]:
from PIL import Image

# Rebuild cleanly, combining both negative sources.
for split in ["train", "val", "test"]:
    for label in ["leaf", "non_leaf"]:
        shutil.rmtree(DATA_DIR / split / label, ignore_errors=True)
        (DATA_DIR / split / label).mkdir(parents=True, exist_ok=True)

# Convert CIFAR images to files.
cifar_dir = Path("/content/cifar_negative_images")
cifar_dir.mkdir(exist_ok=True)

cifar_paths = []
for i, arr in enumerate(negative_arrays):
    path = cifar_dir / f"cifar_{i}.jpg"
    Image.fromarray(arr).save(path, quality=95)
    cifar_paths.append(path)

all_negative_files = cifar_paths + flower_files

def save_pil_images(file_paths, label):
    # Split into train (70%), val (15%), test (15%)
    train_files, val_test_files = train_test_split(file_paths, test_size=0.3, random_state=SEED)
    val_files, test_files = train_test_split(val_test_files, test_size=0.5, random_state=SEED)

    splits = {
        "train": train_files,
        "val": val_files,
        "test": test_files
    }

    for split, files in splits.items():
        dest_dir = DATA_DIR / split / label
        for i, src_path in enumerate(files):
            dest_path = dest_dir / f"{label}_{i}{src_path.suffix}"
            shutil.copy(src_path, dest_path)

save_pil_images(positive_files, "leaf")
save_pil_images(all_negative_files, "non_leaf")

print("Combined positive and negative dataset ready.")

Combined positive and negative dataset ready.


In [6]:
for split in ["train", "val", "test"]:
    print("\n", split)
    for label in ["leaf", "non_leaf"]:
        count = len(list((DATA_DIR / split / label).glob("*.jpg")))
        print(label, count)


 train
leaf 45
non_leaf 1400

 val
leaf 5
non_leaf 300

 test
leaf 4
non_leaf 300


In [7]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,
)

print("Class order:", train_ds.class_names)

Found 3150 files belonging to 2 classes.
Found 675 files belonging to 2 classes.
Found 675 files belonging to 2 classes.
Class order: ['leaf', 'non_leaf']


In [8]:
base_model = MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet",
)

base_model.trainable = False

inputs = layers.Input(shape=(160, 160, 3))

# MobileNetV2 expects pixel values scaled to [-1, 1].
x = layers.Rescaling(1.0 / 127.5, offset=-1)(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

validator = Model(inputs, outputs)

validator.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

validator.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "/content/best_validator.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
    ),
]

history = validator.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
)

Epoch 1/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 37s 190ms/step - accuracy: 0.9927 - auc: 0.9998 - loss: 0.0358 - precision: 0.9935 - recall: 0.9900 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.0025 - val_precision: 1.0000 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 2/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 1.0000 - auc: 1.0000 - loss: 0.0025 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.0012 - val_precision: 1.0000 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 3/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 1.0000 - auc: 1.0000 - loss: 0.0014 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 7.0864e-04 - val_precision: 1.0000 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 4/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 1.0000 - auc: 1.0000 - loss: 0.0011 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 4.5678e-04

In [10]:
validator = tf.keras.models.load_model(
    "/content/best_validator.keras"
)

results = validator.evaluate(test_ds, return_dict=True)
print(results)

y_true = np.concatenate([
    labels.numpy().ravel() for _, labels in test_ds
]).astype(int)

y_score = validator.predict(test_ds, verbose=0).ravel()
y_pred = (y_score >= 0.5).astype(int)

print(classification_report(
    y_true,
    y_pred,
    target_names=["leaf", "non_leaf"],
    zero_division=0,
))

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 137ms/step - accuracy: 1.0000 - auc: 1.0000 - loss: 1.9005e-04 - precision: 1.0000 - recall: 1.0000
{'accuracy': 1.0, 'auc': 1.0, 'loss': 0.00019005026842933148, 'precision': 1.0, 'recall': 1.0}
              precision    recall  f1-score   support

        leaf       1.00      1.00      1.00       375
    non_leaf       1.00      1.00      1.00       300

    accuracy                           1.00       675
   macro avg       1.00      1.00      1.00       675
weighted avg       1.00      1.00      1.00       675

Confusion matrix:
[[375   0]
 [  0 300]]


In [11]:
EXPORT_DIR = Path("/content/validator_export")
EXPORT_DIR.mkdir(exist_ok=True)

validator.save(EXPORT_DIR / "tomato_leaf_validator.keras")

with open(EXPORT_DIR / "validator_labels.json", "w") as f:
    import json
    json.dump(
        {"0": "leaf", "1": "non_leaf"},
        f,
        indent=2,
    )

print("Exported model:", EXPORT_DIR / "tomato_leaf_validator.keras")
print("Exported labels:", EXPORT_DIR / "validator_labels.json")

!cd /content && zip -r tomato_leaf_validator_export.zip validator_export

Exported model: /content/validator_export/tomato_leaf_validator.keras
Exported labels: /content/validator_export/validator_labels.json
  adding: validator_export/ (stored 0%)
  adding: validator_export/validator_labels.json (deflated 17%)
  adding: validator_export/tomato_leaf_validator.keras (deflated 12%)
